============================================================
SPARK INTERNALS - Deep Dive Interview Questions
============================================================
Swiss Re Interview Prep: Senior Application Engineer
Topic: Spark Architecture, Memory, Shuffle, Execution
============================================================


QUESTION: Explain Spark's architecture.

ANSWER:

┌─────────────────────────────────────────────────────────────┐
│                        DRIVER                                │
│  ┌─────────────┐  ┌───────────────┐  ┌────────────────────┐ │
│  │ SparkContext│  │ DAG Scheduler │  │ Task Scheduler     │ │
│  └─────────────┘  └───────────────┘  └────────────────────┘ │
└─────────────────────────────────────────────────────────────┘
                              │
                              ▼
┌─────────────────────────────────────────────────────────────┐
│                   CLUSTER MANAGER                            │
│           (YARN / Kubernetes / Mesos / Standalone)          │
└─────────────────────────────────────────────────────────────┘
                              │
        ┌─────────────────────┼─────────────────────┐
        ▼                     ▼                     ▼
┌──────────────┐     ┌──────────────┐     ┌──────────────┐
│   EXECUTOR   │     │   EXECUTOR   │     │   EXECUTOR   │
│  ┌────────┐  │     │  ┌────────┐  │     │  ┌────────┐  │
│  │  Task  │  │     │  │  Task  │  │     │  │  Task  │  │
│  │  Task  │  │     │  │  Task  │  │     │  │  Task  │  │
│  └────────┘  │     │  └────────┘  │     │  └────────┘  │
│  Cache       │     │  Cache       │     │  Cache       │
└──────────────┘     └──────────────┘     └──────────────┘

KEY COMPONENTS:
- DRIVER: Runs the main() function, creates SparkContext, builds DAG
- CLUSTER MANAGER: Allocates resources across the cluster
- EXECUTOR: JVM process on worker nodes, runs tasks, stores cache
- TASK: Smallest unit of work, runs on one partition



ANSWER: Step-by-step execution:

1. JOB SUBMISSION
   - Your code (transformations + action) is submitted to the Driver

2. DAG CREATION
   - Driver builds a DAG (Directed Acyclic Graph) of stages
   - Stages are separated by shuffles (wide transformations)

3. STAGE DIVISION
   - Each stage contains narrow transformations that can be pipelined
   - A new stage starts after each shuffle boundary

4. TASK SCHEDULING
   - Each stage is divided into tasks (one task per partition)
   - Tasks are sent to executors

5. TASK EXECUTION
   - Executors run tasks in parallel
   - Results are sent back to driver (or written to storage)

EXAMPLE:
df.filter(...).groupBy(...).count().show()

Stage 1: filter (narrow, no shuffle)
-- SHUFFLE BOUNDARY --
Stage 2: groupBy + count (requires shuffle to group by key)
Stage 3: show (collect results to driver)



QUESTION: What is a shuffle and why is it expensive?

ANSWER:
Shuffle = Data redistribution across partitions

WHEN IT HAPPENS:
- groupBy, reduceByKey
- join (unless broadcast)
- repartition
- distinct
- orderBy

WHY IT'S EXPENSIVE:
1. DISK I/O: Map side writes to local disk (shuffle files)
2. NETWORK: Data transferred between executors
3. SERIALIZATION: Data must be serialized/deserialized
4. SORT: Data is often sorted during shuffle

SHUFFLE PROCESS:
┌─────────────┐       Shuffle       ┌─────────────┐
│ Partition 0 │  ─────Write────►   │ Shuffle     │
│ Partition 1 │  ─────Write────►   │ Files       │
│ Partition 2 │  ─────Write────►   │ (on disk)   │
└─────────────┘                     └──────┬──────┘
                                           │
                                       Read│
                                           ▼
                                    ┌─────────────┐
                                    │ Partition A │
                                    │ Partition B │
                                    │ Partition C │
                                    └─────────────┘

HOW TO REDUCE SHUFFLE:
1. Use broadcast joins for small tables
2. Filter data BEFORE shuffle
3. Use salting for skewed keys
4. Increase spark.sql.shuffle.partitions for large data



QUESTION: Explain Spark's memory model.

ANSWER:

EXECUTOR MEMORY BREAKDOWN:
┌────────────────────────────────────────────────────────────┐
│                     EXECUTOR MEMORY                         │
├────────────────────────────────────────────────────────────┤
│  ┌──────────────────────────────────────────────────────┐  │
│  │         UNIFIED MEMORY (spark.memory.fraction)       │  │
│  │                    Default: 0.6                       │  │
│  ├───────────────────────┬──────────────────────────────┤  │
│  │   STORAGE MEMORY      │    EXECUTION MEMORY          │  │
│  │   (Caching RDDs)      │    (Joins, Sorts, Aggs)     │  │
│  │   Can borrow from     │    Can borrow from          │  │
│  │   Execution           │    Storage (evicts cache)   │  │
│  └───────────────────────┴──────────────────────────────┘  │
├────────────────────────────────────────────────────────────┤
│           USER MEMORY (1 - spark.memory.fraction)          │
│           (User data structures, UDF variables)            │
├────────────────────────────────────────────────────────────┤
│           RESERVED MEMORY (300MB fixed)                    │
│           (Spark internal)                                  │
└────────────────────────────────────────────────────────────┘

COMMON OOM CAUSES:
1. Too many partitions for available memory
2. Skewed data (one partition too big)
3. Too much data cached
4. UDFs holding large objects

TUNING:
spark.executor.memory = 8g
spark.memory.fraction = 0.6
spark.memory.storageFraction = 0.5



QUESTION: When should you cache, and what storage level should you use?

ANSWER:

WHEN TO CACHE:
- DataFrame/RDD used multiple times
- After expensive computation (join, aggregation)
- Iterative algorithms

STORAGE LEVELS:
┌──────────────────┬─────────┬─────────┬────────────────────┐
│ Level            │ Memory  │ Disk    │ Notes              │
├──────────────────┼─────────┼─────────┼────────────────────┤
│ MEMORY_ONLY      │ ✓       │         │ Fast, may OOM      │
│ MEMORY_AND_DISK  │ ✓       │ ✓       │ Recommended        │
│ DISK_ONLY        │         │ ✓       │ Slow but safe      │
│ MEMORY_ONLY_SER  │ ✓ (ser) │         │ Less memory, CPU   │
│ MEMORY_AND_DISK  │ ✓ (ser) │ ✓       │ Balanced           │
│ OFF_HEAP         │ ✓ (off) │         │ Avoid GC pauses    │
└──────────────────┴─────────┴─────────┴────────────────────┘

CODE:
from pyspark.storagelevel import StorageLevel

df.cache()  # Same as persist(MEMORY_AND_DISK)
df.persist(StorageLevel.MEMORY_AND_DISK_SER)
df.unpersist()  # Remove from cache

GOTCHA:
Caching is LAZY! It happens only when an action is called.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.storagelevel import StorageLevel


In [ ]:
def caching_demo():
    spark = SparkSession.builder.appName("CacheDemo").master("local[*]").getOrCreate()
    
    df = spark.range(0, 1000000)
    
    # This does NOT cache immediately
    df.cache()
    
    # First action triggers caching
    count1 = df.count()  # Slow (computes and caches)
    
    # Second action uses cache
    count2 = df.count()  # Fast (reads from cache)
    
    # Clean up
    df.unpersist()
    spark.stop()


In [ ]:
caching_demo()


QUESTION: What are broadcast variables and when do you use them?

ANSWER:
Broadcast = Send read-only data to ALL executors ONCE.

WITHOUT BROADCAST:
- Data is serialized with each TASK
- 1000 tasks = 1000 copies sent over network

WITH BROADCAST:
- Data is sent ONCE per executor
- 10 executors = 10 copies (not 1000)

USE CASES:
- Small lookup tables in joins
- Configuration dictionaries
- Machine learning models


In [ ]:
def broadcast_demo():
    spark = SparkSession.builder.appName("BroadcastDemo").master("local[*]").getOrCreate()
    
    # Small lookup table
    country_codes = {"US": "United States", "UK": "United Kingdom", "IN": "India"}
    
    # Broadcast it
    broadcast_codes = spark.sparkContext.broadcast(country_codes)
    
    # Use in UDF
    from pyspark.sql.functions import udf
    from pyspark.sql.types import StringType
    
    @udf(StringType())
    def get_country_name(code):
        return broadcast_codes.value.get(code, "Unknown")
    
    # The broadcast variable is available on all executors
    
    spark.stop()


In [ ]:
broadcast_demo()


QUESTION: What are accumulators and what's the gotcha?

ANSWER:
Accumulators = Variables that workers can ADD to (not read).

USE CASES:
- Counting events (errors, records processed)
- Summing values across partitions

GOTCHA:
- Only reliable in ACTIONS, not transformations
- If a task is retried, the accumulator is incremented AGAIN!
- Use only for debugging/monitoring, not for business logic


In [ ]:
def accumulator_demo():
    spark = SparkSession.builder.appName("AccumulatorDemo").master("local[*]").getOrCreate()
    sc = spark.sparkContext
    
    # Create accumulator
    error_count = sc.accumulator(0)
    
    def process_row(row):
        if row < 0:
            error_count.add(1)
        return row * 2
    
    rdd = sc.parallelize([-1, 2, -3, 4, 5])
    result = rdd.map(process_row).collect()
    
    print(f"Error count: {error_count.value}")  # 2
    
    spark.stop()


In [ ]:
accumulator_demo()


QUESTION: What is AQE and what problems does it solve?

ANSWER:
AQE (Adaptive Query Execution) optimizes queries AT RUNTIME based on actual data statistics.

FEATURES:
1. COALESCING SHUFFLE PARTITIONS
   - Automatically merges small partitions after shuffle
   - Solves "too many small files" problem

2. CONVERTING SORT-MERGE TO BROADCAST JOIN
   - If one side of join is small after filtering, broadcast it
   - Avoids expensive sort-merge join

3. SKEW JOIN OPTIMIZATION
   - Detects skewed partitions at runtime
   - Splits skewed partitions into smaller chunks

ENABLE:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")



QUESTION: How does Catalyst optimize queries?

ANSWER:

PHASES:
1. ANALYSIS: Resolve table/column names, check types
2. LOGICAL OPTIMIZATION: Apply rule-based optimizations
3. PHYSICAL PLANNING: Generate multiple physical plans
4. CODE GENERATION: Generate Java bytecode (Tungsten)

KEY OPTIMIZATIONS:
- PREDICATE PUSHDOWN: Push filters to data source
- PROJECTION PRUNING: Read only needed columns
- CONSTANT FOLDING: Evaluate constant expressions at compile time
- JOIN REORDERING: Put smaller tables first

EXAMPLE:
df.filter(col("x") > 10).filter(col("x") < 100)

BECOMES:
df.filter((col("x") > 10) & (col("x") < 100))  # Combined into one filter

SEE THE PLAN:
df.explain(True)  # Shows all optimization phases



| Problem               | Symptom                    | Fix                           |
|-----------------------|----------------------------|-------------------------------|
| Data Skew             | One task takes forever     | Salting, AQE, broadcast       |
| Too Many Partitions   | Many small files           | coalesce(), AQE               |
| Too Few Partitions    | OOM, slow processing       | repartition()                 |
| Shuffle Too Large     | High network I/O           | Filter early, broadcast join  |
| No Caching            | Re-computation             | cache() for reused DFs        |
| Driver OOM            | collect() on big data      | Use write() instead           |
| Executor OOM          | Complex UDFs, wide rows    | Increase memory, reduce skew  |
| Serialization Issues  | Python UDF slow            | Use Pandas UDF, built-in fns  |


In [ ]:
if __name__ == "__main__":
    print("Spark Internals - Study Guide")
    print("=" * 60)
    print("Run each demo function to see the concepts in action.")
    print("Use df.explain(True) to see query plans.")
